# LeetCode 687 — Longest Univalue Path bằng Python

Link đề: https://leetcode.com/problems/longest-univalue-path/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree, Tree DP

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def longestUnivaluePath(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: LeetCode tự cung cấp `TreeNode`; trong notebook ta **tự dựng** `TreeNode` cùng hàm dựng cây từ danh sách kiểu LeetCode (`[5,4,5,1,1,null,5]`) để chạy thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong cây nhị phân, tìm **đường đi dài nhất** mà **mọi nút có cùng giá trị**; độ dài đo bằng **số cạnh**; đường đi có thể **không qua gốc** và có thể **gấp khúc** tại một nút (đi xuống trái rồi đi xuống phải).
2. Nắm các công cụ: cấu trúc `TreeNode`, đọc/ghi cây dạng danh sách theo tầng (`None` là nút rỗng), duyệt **hậu thứ tự** bằng đệ quy, `nonlocal`, hàng đợi `deque`, giới hạn đệ quy của Python.
3. Đi từ **xét mọi cặp nút** → **xét từng nút làm "đỉnh" của đường đi** → **DFS trả về chuỗi dài nhất đi xuống** (hai cách mô tả) → **bản lặp (không đệ quy) bằng thứ tự BFS đảo ngược**.
4. Hiểu mấu chốt: mỗi đường đi có **một nút cao nhất** (đỉnh); với mỗi nút ta **nối hai chuỗi giá trị bằng nhau đi xuống** (trái + phải) để cập nhật đáp án, nhưng chỉ **trả lên cha một nhánh** (`max(trái, phải)`) vì đường đi không được rẽ nhánh.
5. Tránh các lỗi kinh điển: đếm nút thay vì cạnh, bỏ qua trường hợp gấp khúc, quên kiểm tra giá trị con bằng cha, trả `l + r` lên cha, và chỉ tính đường đi qua gốc.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân. Trả về **độ dài đường đi dài nhất** mà **mọi nút trên đường đi có cùng giá trị**. Đường đi có thể đi qua hoặc không qua gốc. **Độ dài đường đi giữa hai nút là số cạnh** giữa chúng.

**Ví dụ 1:** `root = [5,4,5,1,1,null,5]` ⇒ `2`.

```
        5
       / \
      4   5
     / \   \
    1   1   5
```

Đường đi dài nhất cùng giá trị là `5 - 5 - 5` (gốc, con phải, cháu phải): 3 nút, **2 cạnh**.

**Ví dụ 2:** `root = [1,4,5,4,4,null,5]` ⇒ `2`.

```
        1
       / \
      4   5
     / \   \
    4   4   5
```

Đường đi `4 - 4 - 4` (con trái của gốc cùng hai con của nó) gấp khúc tại nút `4`: 2 cạnh. (Đường `5 - 5` chỉ có 1 cạnh.)

**Ràng buộc:**

- Số nút trong `[0, 10^4]` (có thể là cây rỗng).
- `-1000 <= Node.val <= 1000`.
- Độ sâu cây không vượt quá `1000`.

Cell dưới dựng `TreeNode`, hàm `build_tree`/`tree_to_list` (danh sách kiểu LeetCode) và hàm sinh cây ngẫu nhiên; rồi in thử hai ví dụ.

In [1]:
from collections import deque
import random
import sys
import time
from itertools import combinations
from typing import List, Optional

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

In [3]:
def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)

In [4]:
def show(root, indent=0, tag="root"):
    """In cây xoay ngang: mỗi nút một dòng, thụt lề theo độ sâu."""
    if root is None:
        return
    print("    " * indent + f"{tag}: {root.val}")
    show(root.left, indent + 1, "L")
    show(root.right, indent + 1, "R")


ex1 = build_tree([5, 4, 5, 1, 1, None, 5])
ex2 = build_tree([1, 4, 5, 4, 4, None, 5])
show(ex1)
print("---")
show(ex2)
print(tree_to_list(ex1), tree_to_list(ex2))          # đọc ngược lại ra đúng danh sách ban đầu
assert tree_to_list(ex1) == [5, 4, 5, 1, 1, None, 5] and tree_to_list(ex2) == [1, 4, 5, 4, 4, None, 5]

root: 5
    L: 4
        L: 1
        R: 1
    R: 5
        R: 5
---
root: 1
    L: 4
        L: 4
        R: 4
    R: 5
        R: 5
[5, 4, 5, 1, 1, None, 5] [1, 4, 5, 4, 4, None, 5]


## 3. Kiến thức nền

### 3.1. Cây nhị phân và `TreeNode`

Mỗi nút có `val` (giá trị), `left` (con trái), `right` (con phải); `None` nghĩa là **không có** con. Nút không có con nào gọi là **lá**. **Cạnh** là đoạn nối nút cha với nút con. Một đường đi gồm `k` nút có `k - 1` cạnh.

In [5]:
root = TreeNode(1, TreeNode(2), TreeNode(3))
print(root.val, root.left.val, root.right.val, root.left.left)      # 1 2 3 None
leaf = root.left
print("lá?", leaf.left is None and leaf.right is None)               # True

1 2 3 None
lá? True


### 3.2. Danh sách kiểu LeetCode (duyệt theo tầng, `None` là nút rỗng)

`[5,4,5,1,1,null,5]` đọc theo **từng tầng, từ trái sang phải**: tầng 0 là `5`; tầng 1 là `4, 5`; tầng 2 là con của `4` (`1, 1`) rồi con của `5` (`null, 5`). `build_tree` dùng một hàng đợi `deque`: lấy nút ra, gán hai giá trị kế tiếp làm con trái/phải.

Lưu ý quy ước: nút `None` **không** có con trong danh sách (không ghi thêm hai `null` cho nó).

In [6]:
t = build_tree([1, None, 2, 3])
show(t)
print(tree_to_list(t))
print(build_tree([]), tree_to_list(None))

root: 1
    R: 2
        L: 3
[1, None, 2, 3]
None []


### 3.3. Duyệt hậu thứ tự (post-order) và đệ quy trả giá trị

**Hậu thứ tự:** xử lý **con trái, con phải trước, rồi mới nút hiện tại**. Rất hợp để "tổng hợp thông tin từ con lên cha". Mẫu: hàm `dfs(node)` gọi `dfs` cho hai con, nhận giá trị trả về, rồi tính giá trị của `node`. Ví dụ tính **chiều cao** của cây (số cạnh của đường dài nhất xuống lá).

In [7]:
def height(node):
    """Số cạnh trên đường dài nhất từ node xuống một lá (cây rỗng: -1, lá: 0)."""
    if node is None:
        return -1
    return 1 + max(height(node.left), height(node.right))


print(height(ex1), height(None), height(TreeNode(7)))        # 2 -1 0

2 -1 0


### 3.4. `nonlocal` — cập nhật biến của hàm ngoài từ hàm trong

Khi hàm `dfs` lồng trong hàm chính và muốn **gán lại** một biến số của hàm ngoài (như `ans`), phải khai báo `nonlocal ans`; nếu không Python hiểu `ans = ...` là biến cục bộ mới.

In [8]:
def count_nodes(root):
    total = 0

    def dfs(node):
        nonlocal total                      # nếu thiếu dòng này: UnboundLocalError
        if node is None:
            return
        total += 1
        dfs(node.left)
        dfs(node.right)

    dfs(root)
    return total


print(count_nodes(ex1), count_nodes(None))   # 7 0

6 0


### 3.5. Giới hạn đệ quy của Python

Mặc định Python cho phép đệ quy sâu khoảng `1000` tầng; cây lệch có thể sâu tới `1000` (theo đề) nên đệ quy dễ chạm giới hạn. LeetCode nâng giới hạn; trong notebook ta tự nâng bằng `sys.setrecursionlimit`. Cách an toàn hơn: viết **bản lặp** (Cách 4).

In [9]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
sys.setrecursionlimit(10000)               # nâng lên cho các test cây sâu ở mục 12
print("sau khi nâng:", sys.getrecursionlimit())

giới hạn đệ quy mặc định: 1000
sau khi nâng: 10000


### 3.6. Đường đi trong cây, "đỉnh" của đường đi và số cạnh

Giữa hai nút bất kỳ của cây có **đúng một** đường đi đơn. Đường đi đó đi **lên** từ một nút đến nút **tổ tiên chung thấp nhất** (đỉnh), rồi đi **xuống** nút kia. Đường đi `4 - 4 - 4` ở ví dụ 2 có đỉnh là nút `4` ở giữa: đi xuống trái một cạnh và xuống phải một cạnh (tổng 2 cạnh). Đường đi chỉ gồm một nút có độ dài `0`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1 (`[5,4,5,1,1,null,5]`):

- Nhóm các nút giá trị `5`: gốc, con phải của gốc, cháu phải (con phải của con phải). Chúng nối với nhau thành một chuỗi **gốc → con phải → cháu phải**: 3 nút, 2 cạnh.
- Nút `4` có hai con `1` và `1` (khác `4`): không nối được.
- Hai nút `1` là lá, không nối với nhau (chúng là anh em, đi qua `4` sẽ lẫn giá trị).

Đáp án = **số cạnh lớn nhất** trong các "đám" cùng giá trị nối liền = `2`.

Ví dụ 2: các nút `4` (con trái của gốc và hai con của nó) tạo đám hình chữ V: ở giữa nút `4`, hai nhánh dưới là hai `4`. Đường đi dài nhất trong đám là **trái–giữa–phải** (gấp khúc): 2 cạnh. Đám `5` (con phải gốc + con phải của nó) chỉ có 1 cạnh.

**Cách nghĩ xuyên suốt:** nối một nút với con **chỉ khi hai nút có cùng giá trị**. Nhìn từ một nút `x` làm "đỉnh":

- nhánh trái cho ta một chuỗi đi xuống dài `L` cạnh (nếu con trái cùng giá trị),
- nhánh phải cho `R` cạnh,
- đường đi gấp khúc tại `x` dài **`L + R`**.

Đáp án là max của `L + R` trên mọi nút `x`.

## 5. Cách 1 — Brute Force / baseline: xét mọi cặp nút

**Ý tưởng bám sát định nghĩa:** một đường đi trong cây hoàn toàn được xác định bởi **hai đầu mút** (hai nút `u`, `v`). Với mọi cặp `u ≠ v`:

1. Tìm đường đi giữa `u` và `v` (leo lên cha từ cả hai đầu cho đến **tổ tiên chung**).
2. Nếu **mọi nút trên đường đi có cùng giá trị** thì độ dài = số nút − 1; cập nhật đáp án.
3. Đường đi một nút có độ dài 0 (đáp án tối thiểu là `0`).

Để leo lên cha ta dựng bảng `parent` bằng một lần duyệt. Đây là **oracle** độc lập (không dùng tư duy DP) để đối chiếu ở mục 12.

In [10]:
def lup_bruteforce(root):
    """Cách 1: xét mọi cặp (u, v), dựng đường đi qua tổ tiên chung, kiểm tra cùng giá trị."""
    if root is None:
        return 0
    parent, nodes = {root: None}, []
    stack = [root]
    while stack:
        node = stack.pop()
        nodes.append(node)
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)

    def chain_to_root(u):
        out = []
        while u is not None:
            out.append(u)
            u = parent[u]
        return out

    best = 0
    for u, v in combinations(nodes, 2):
        cu, cv = chain_to_root(u), chain_to_root(v)
        set_v = set(cv)
        i = next(i for i, x in enumerate(cu) if x in set_v)        # tổ tiên chung thấp nhất = cu[i]
        j = cv.index(cu[i])
        path = cu[:i + 1] + cv[:j]                                  # u ... lca ... v (lca chỉ một lần)
        if len({x.val for x in path}) == 1:
            best = max(best, len(path) - 1)
    return best


print(lup_bruteforce(ex1), lup_bruteforce(ex2), lup_bruteforce(None), lup_bruteforce(TreeNode(3)))   # 2 2 0 0
assert lup_bruteforce(ex1) == 2 and lup_bruteforce(ex2) == 2

2 2 0 0


**Độ phức tạp:** có `n(n-1)/2` cặp; mỗi cặp leo cây tốn `O(h)` (`h` = chiều cao) và kiểm tra đường đi `O(h)` ⇒ **`O(n² · h)`** thời gian; bộ nhớ `O(n)` cho bảng `parent`. Với `n = 10^4` và `h` lớn ⇒ cỡ `10^{11}` — không thể.

**Nút thắt:** ta xét **quá nhiều đường đi**. Có tới `O(n²)` đường đi nhưng mỗi đường đi có **một đỉnh duy nhất**; nếu duyệt theo đỉnh (mục 6) chỉ có `n` ứng viên, và với mỗi đỉnh ta chỉ cần biết **chuỗi dài nhất đi xuống** ở hai bên.

## 6. Ý tưởng thuật toán / bước đột phá

Mỗi đường đi có đúng **một nút cao nhất** (đỉnh). Gọi `x` là đỉnh:

- Đường đi = (chuỗi đi xuống từ `x` về trái) + (chuỗi đi xuống từ `x` về phải); mỗi chuỗi chỉ gồm các nút **cùng giá trị với `x`**.
- Vậy độ dài (số cạnh) = `L(x) + R(x)`, với:
  - `L(x) = 1 + down(x.left)` nếu `x.left` tồn tại **và** `x.left.val == x.val`, ngược lại `0`;
  - `R(x)` tương tự với con phải;
  - `down(y)` = **số cạnh của chuỗi dài nhất cùng giá trị đi xuống bắt đầu từ `y`** = `max(L(y), R(y))`.

Công thức đệ quy gọn: `dfs(x)` trả về `down(x) = max(L(x), R(x))`, và **trong lúc tính** cập nhật `ans = max(ans, L(x) + R(x))`.

**Điểm tinh tế:** trả lên cho cha **chỉ `max(L, R)`**, không phải `L + R` — vì cha chỉ có thể nối với **một** nhánh của `x` (đường đi không rẽ nhánh). `L + R` chỉ dùng để cập nhật đáp án tại đỉnh `x`.

Mỗi nút được xử lý một lần ⇒ **`O(n)`**. Ta sẽ lần lượt: Cách 2 (DFS trả về số cạnh — chính thức), Cách 3 (DFS có tham số giá trị cha, đếm nút), Cách 4 (bản lặp, không đệ quy).

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: mọi cặp nút (mục 5, oracle).
- Cách 2: DFS hậu thứ tự, trả về số cạnh (**chính thức**).
- Cách 3: DFS có tham số "giá trị của cha", trả về số nút của chuỗi khớp.
- Cách 4: bản lặp bằng thứ tự BFS đảo ngược (không đệ quy).

---

### Cách 2 — DFS trả về chuỗi cùng giá trị dài nhất đi xuống (số cạnh)

Giải thích từng đoạn:

1. `dfs(node)`: nếu `node is None` trả `0`.
2. `l = dfs(node.left)`, `r = dfs(node.right)`: chuỗi dài nhất (số cạnh) đi xuống **bắt đầu từ con**.
3. Nếu có con trái **và** `node.left.val == node.val` thì `l = l + 1` (thêm cạnh nối node với con trái); ngược lại `l = 0`. Tương tự `r`.
4. `ans = max(ans, l + r)` (đường gấp khúc có đỉnh tại `node`).
5. Trả `max(l, r)` lên cha.

In [11]:
def lup_dfs(root):
    """Cách 2: dfs(node) = số cạnh của chuỗi cùng giá trị dài nhất đi xuống từ node."""
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l = dfs(node.left)
        r = dfs(node.right)
        l = l + 1 if node.left and node.left.val == node.val else 0
        r = r + 1 if node.right and node.right.val == node.val else 0
        ans = max(ans, l + r)
        return max(l, r)

    dfs(root)
    return ans


print(lup_dfs(ex1), lup_dfs(ex2), lup_dfs(None))
assert lup_dfs(ex1) == 2 and lup_dfs(ex2) == 2 and lup_dfs(None) == 0

2 2 0


**Code nộp LeetCode cho Cách 2:**

```python
class Solution:
    def longestUnivaluePath(self, root: Optional[TreeNode]) -> int:
        def dfs(root):
            if root is None:
                return 0
            l, r = dfs(root.left), dfs(root.right)
            l = l + 1 if root.left and root.left.val == root.val else 0
            r = r + 1 if root.right and root.right.val == root.val else 0
            nonlocal ans
            ans = max(ans, l + r)
            return max(l, r)

        ans = 0
        dfs(root)
        return ans
```

**Độ phức tạp:** thời gian `O(n)` (mỗi nút một lần), bộ nhớ `O(h)` cho ngăn xếp đệ quy (`h ≤ 1000`).

---

### Cách 3 — DFS có tham số "giá trị của cha", trả về số nút

Cách mô tả khác (cùng bản chất): hàm `dfs(node, parent_val)` trả về **số nút** của chuỗi dài nhất bắt đầu từ `node` và đi xuống, **với điều kiện `node.val == parent_val`**; nếu không khớp trả `0`.

Giải thích từng đoạn:

1. `node is None` ⇒ `0`.
2. Gọi `l = dfs(node.left, node.val)` và `r = dfs(node.right, node.val)`: đây là số nút của chuỗi con (đã **tự kiểm tra khớp** với `node.val`).
3. `ans = max(ans, l + r)`: `l` nút bên trái + `r` nút bên phải, cộng với nút `node` ở giữa là `l + r + 1` nút ⇒ `l + r` **cạnh**.
4. Nếu `node.val == parent_val` thì trả `1 + max(l, r)` (nút `node` cộng chuỗi dài hơn); ngược lại `0`.
5. Gọi `dfs(root, None)` (gốc không có cha).

Ưu điểm: kiểm tra "khớp với cha" nằm ngay đầu hàm, không phải viết `node.left and node.left.val == node.val` hai lần.

In [12]:
def lup_parent_val(root):
    """Cách 3: dfs(node, parent_val) = số NÚT của chuỗi khớp parent_val bắt đầu ở node (0 nếu không khớp)."""
    ans = 0

    def dfs(node, parent_val):
        nonlocal ans
        if node is None:
            return 0
        l = dfs(node.left, node.val)
        r = dfs(node.right, node.val)
        ans = max(ans, l + r)                       # l + r CẠNH (có thêm nút giữa: l + r + 1 nút)
        return 1 + max(l, r) if node.val == parent_val else 0

    dfs(root, None)
    return ans


print(lup_parent_val(ex1), lup_parent_val(ex2), lup_parent_val(None))
assert lup_parent_val(ex1) == 2 and lup_parent_val(ex2) == 2 and lup_parent_val(None) == 0

2 2 0


**Code nộp LeetCode cho Cách 3:**

```python
class Solution:
    def longestUnivaluePath(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node, parent_val):
            nonlocal ans
            if node is None:
                return 0
            l = dfs(node.left, node.val)
            r = dfs(node.right, node.val)
            ans = max(ans, l + r)
            return 1 + max(l, r) if node.val == parent_val else 0

        dfs(root, None)
        return ans
```

**Độ phức tạp:** `O(n)` thời gian, `O(h)` bộ nhớ. (Lưu ý: giá trị `parent_val = None` cho gốc không bao giờ bằng `node.val` nguyên.)

---

### Cách 4 — Bản lặp: thứ tự BFS đảo ngược (không đệ quy)

Vì đệ quy sâu có thể chạm giới hạn của Python, ta thay bằng vòng lặp:

1. Duyệt **BFS** (hàng đợi) ghi lại `order` = danh sách nút theo tầng (cha **luôn đứng trước** con).
2. Xử lý `order` **từ cuối về đầu**: khi gặp một nút thì **cả hai con đã được xử lý rồi**.
3. Giữ từ điển `down[node]` = số cạnh chuỗi dài nhất đi xuống từ `node`; tính `l`, `r` như Cách 2 từ `down` của hai con; cập nhật `ans`; ghi `down[node] = max(l, r)`.

In [13]:
def lup_iterative(root):
    """Cách 4: không đệ quy. order = thứ tự BFS; xử lý ngược để con luôn được tính trước cha."""
    if root is None:
        return 0
    order = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        order.append(node)
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)

    down, ans = {}, 0
    for node in reversed(order):
        l = down[node.left] + 1 if node.left and node.left.val == node.val else 0
        r = down[node.right] + 1 if node.right and node.right.val == node.val else 0
        ans = max(ans, l + r)
        down[node] = max(l, r)
    return ans


print(lup_iterative(ex1), lup_iterative(ex2), lup_iterative(None))
assert lup_iterative(ex1) == 2 and lup_iterative(ex2) == 2 and lup_iterative(None) == 0

2 2 0


**Code nộp LeetCode cho Cách 4:**

```python
from collections import deque

class Solution:
    def longestUnivaluePath(self, root: Optional[TreeNode]) -> int:
        if root is None:
            return 0
        order, queue = [], deque([root])
        while queue:
            node = queue.popleft()
            order.append(node)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        down, ans = {}, 0
        for node in reversed(order):
            l = down[node.left] + 1 if node.left and node.left.val == node.val else 0
            r = down[node.right] + 1 if node.right and node.right.val == node.val else 0
            ans = max(ans, l + r)
            down[node] = max(l, r)
        return ans
```

**Độ phức tạp:** `O(n)` thời gian; bộ nhớ `O(n)` cho `order` và `down` (nhưng **không phụ thuộc giới hạn đệ quy**, an toàn với cây rất sâu).

## 8. TRACE từng bước

### 8.1. Trace Cách 2 với ví dụ 1 `[5,4,5,1,1,null,5]`

Thứ tự hậu thứ tự (xử lý nút sau cùng): lá `1` (trái), lá `1` (phải), nút `4`, lá `5` (cháu), nút `5` (con phải), gốc `5`.

| nút | l (sau khi kiểm tra bằng cha) | r | `ans` sau bước | trả lên |
|---|---|---|---|---|
| lá `1` (trái của 4) | 0 | 0 | 0 | 0 |
| lá `1` (phải của 4) | 0 | 0 | 0 | 0 |
| `4` | con `1 ≠ 4` ⇒ 0 | con `1 ≠ 4` ⇒ 0 | 0 | 0 |
| lá `5` (cháu) | 0 | 0 | 0 | 0 |
| `5` (con phải của gốc) | không có con trái ⇒ 0 | con `5 = 5` ⇒ `0 + 1 = 1` | `max(0, 0+1) = 1` | 1 |
| gốc `5` | con `4 ≠ 5` ⇒ 0 | con `5 = 5` ⇒ `1 + 1 = 2` | `max(1, 0+2) = 2` | 2 |

Đáp án `2` ✓. Hàm `trace_dfs` bên dưới in từng bước (thụt lề theo độ sâu).

In [14]:
def trace_dfs(root):
    ans = 0

    def dfs(node, depth):
        nonlocal ans
        pad = "  " * depth
        if node is None:
            return 0
        print(f"{pad}vào nút {node.val}")
        l0 = dfs(node.left, depth + 1)
        r0 = dfs(node.right, depth + 1)
        l = l0 + 1 if node.left and node.left.val == node.val else 0
        r = r0 + 1 if node.right and node.right.val == node.val else 0
        old = ans
        ans = max(ans, l + r)
        print(f"{pad}ra nút {node.val}: từ con trái {l0}->{l}, từ con phải {r0}->{r}, "
              f"ans {old}->{ans}, trả lên {max(l, r)}")
        return max(l, r)

    dfs(root, 0)
    return ans


assert trace_dfs(ex1) == 2

vào nút 5
  vào nút 4
    vào nút 1
    ra nút 1: từ con trái 0->0, từ con phải 0->0, ans 0->0, trả lên 0
    vào nút 1
    ra nút 1: từ con trái 0->0, từ con phải 0->0, ans 0->0, trả lên 0
  ra nút 4: từ con trái 0->0, từ con phải 0->0, ans 0->0, trả lên 0
  vào nút 5
    vào nút 5
    ra nút 5: từ con trái 0->0, từ con phải 0->0, ans 0->0, trả lên 0
  ra nút 5: từ con trái 0->0, từ con phải 0->1, ans 0->1, trả lên 1
ra nút 5: từ con trái 0->0, từ con phải 1->2, ans 1->2, trả lên 2


### 8.2. Trace ví dụ 2 `[1,4,5,4,4,null,5]` (đường đi **gấp khúc**)

- Hai lá `4` (con của nút `4` bên trái): trả `0`.
- Nút `4` (con trái của gốc): con trái `4 = 4` ⇒ `l = 0 + 1 = 1`; con phải `4 = 4` ⇒ `r = 0 + 1 = 1`; **`ans = max(0, 1 + 1) = 2`** (đường gấp khúc `4–4–4`); trả lên `max(1, 1) = 1`.
- Lá `5` (cháu): `0`. Nút `5` (con phải của gốc): `r = 0 + 1 = 1`, `ans = max(2, 1) = 2`, trả `1`.
- Gốc `1`: hai con `4`, `5` đều khác `1` ⇒ `l = r = 0`; `ans` giữ `2`.

Chú ý: nếu chỉ trả `max(l, r)` mà **không** cập nhật `ans` bằng `l + r` tại nút `4` thì ta chỉ thu được `1`. Hàm trace bên dưới in cho cây này.

In [15]:
assert trace_dfs(ex2) == 2

vào nút 1
  vào nút 4
    vào nút 4
    ra nút 4: từ con trái 0->0, từ con phải 0->0, ans 0->0, trả lên 0
    vào nút 4
    ra nút 4: từ con trái 0->0, từ con phải 0->0, ans 0->0, trả lên 0
  ra nút 4: từ con trái 0->1, từ con phải 0->1, ans 0->2, trả lên 1
  vào nút 5
    vào nút 5
    ra nút 5: từ con trái 0->0, từ con phải 0->0, ans 2->2, trả lên 0
  ra nút 5: từ con trái 0->0, từ con phải 0->1, ans 2->2, trả lên 1
ra nút 1: từ con trái 1->0, từ con phải 1->0, ans 2->2, trả lên 0


### 8.3. Trace Cách 4 (BFS đảo ngược) với ví dụ 1

`order` (BFS) = `[5(gốc), 4, 5, 1, 1, 5]`. Xử lý ngược: `5(cháu)`, `1`, `1`, `5(con phải)`, `4`, gốc. Từ điển `down` được điền dần; kết quả giống bảng 8.1.

In [16]:
def trace_iterative(root):
    order, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        order.append(node)
        for child in (node.left, node.right):
            if child:
                queue.append(child)
    print("order (BFS):", [n.val for n in order])
    down, ans = {}, 0
    for idx in range(len(order) - 1, -1, -1):
        node = order[idx]
        l = down[node.left] + 1 if node.left and node.left.val == node.val else 0
        r = down[node.right] + 1 if node.right and node.right.val == node.val else 0
        ans = max(ans, l + r)
        down[node] = max(l, r)
        print(f"order[{idx}] giá trị {node.val}: l={l} r={r} -> down={down[node]}, ans={ans}")
    return ans


assert trace_iterative(ex1) == 2

order (BFS): [5, 4, 5, 1, 1, 5]
order[5] giá trị 5: l=0 r=0 -> down=0, ans=0
order[4] giá trị 1: l=0 r=0 -> down=0, ans=0
order[3] giá trị 1: l=0 r=0 -> down=0, ans=0
order[2] giá trị 5: l=0 r=1 -> down=1, ans=1
order[1] giá trị 4: l=0 r=0 -> down=0, ans=1
order[0] giá trị 5: l=0 r=2 -> down=2, ans=2


## 9. Code nộp LeetCode

Cách 2 (chính thức): đúng `class Solution`, phương thức `longestUnivaluePath(self, root: Optional[TreeNode]) -> int`.

In [17]:
class Solution:
    def longestUnivaluePath(self, root: Optional[TreeNode]) -> int:
        def dfs(root):
            if root is None:
                return 0
            l, r = dfs(root.left), dfs(root.right)
            l = l + 1 if root.left and root.left.val == root.val else 0
            r = r + 1 if root.right and root.right.val == root.val else 0
            nonlocal ans
            ans = max(ans, l + r)
            return max(l, r)

        ans = 0
        dfs(root)
        return ans


sol = Solution()
print(sol.longestUnivaluePath(build_tree([5, 4, 5, 1, 1, None, 5])))     # 2
print(sol.longestUnivaluePath(build_tree([1, 4, 5, 4, 4, None, 5])))     # 2
print(sol.longestUnivaluePath(build_tree([])))                           # 0

2
2
0


## 10. Vì sao đúng

1. **Mọi đường đi có đúng một đỉnh `x`:** đường đi đơn trong cây = đi lên từ một đầu mút tới tổ tiên chung thấp nhất `x`, rồi đi xuống tới đầu mút kia. Nên mọi đường đi hợp lệ được "thấy" đúng một lần khi ta xét đỉnh `x` của nó.
2. **Định nghĩa `dfs(x)`:** trả số cạnh của chuỗi **cùng giá trị với `x`** dài nhất **đi xuống bắt đầu từ `x`**. Bằng quy nạp theo chiều cao:
   - Cây rỗng: `0`, và lá: `l = r = 0` ⇒ `0` (chuỗi chỉ có một nút) ✓.
   - Nút `x`: chuỗi dài nhất đi xuống từ `x` phải đi qua con trái hoặc con phải (hoặc dừng tại `x`). Đi qua con trái chỉ hợp lệ nếu `x.left.val == x.val`, khi đó dài `1 + dfs(x.left)`; (chú ý `dfs(x.left)` là chuỗi cùng giá trị với **`x.left`** = cùng giá trị với `x`). Vậy `dfs(x) = max(l, r)` ✓.
3. **Cập nhật đáp án:** đường đi cùng giá trị có đỉnh `x` gồm hai chuỗi đi xuống (trái/phải) cùng giá trị với `x`, dài tối đa `l + r` cạnh (chọn chuỗi dài nhất ở mỗi phía — hai phía độc lập). Ta cập nhật `ans = max(ans, l + r)` ⇒ `ans` = max trên mọi đỉnh ⇒ đúng đáp án.
4. **Trả `max(l, r)` chứ không `l + r`:** khi đi lên cha, đường đi phải **đi tiếp thẳng**, không thể rẽ nhánh vào cả hai con của `x`.
5. **Đường đi một nút:** độ dài `0` ⇒ `ans` khởi tạo `0` đúng.
6. **Cách 3** là cùng ý nhưng đếm theo nút (đưa kiểm tra "bằng cha" vào đầu hàm); **Cách 4** tính cùng công thức theo thứ tự "con trước cha".
7. Đối chiếu thực nghiệm với oracle mọi-cặp (Cách 1) ở mục 12.

## 11. Phân tích Time / Space Complexity

`n` = số nút, `h` = chiều cao cây.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp nút | `O(n² · h)` | `O(n)` | oracle, chỉ cho cây nhỏ |
| 2. DFS trả số cạnh | `O(n)` | `O(h)` | **nộp bài** |
| 3. DFS có `parent_val` | `O(n)` | `O(h)` | cùng ý, đếm nút |
| 4. Bản lặp (BFS đảo ngược) | `O(n)` | `O(n)` | không giới hạn đệ quy |

`h` nằm giữa `log₂ n` (cây cân bằng) và `n` (cây lệch); đề cho `h ≤ 1000`, đệ quy vẫn khả thi.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (cả 4 cách + `Solution`).
2. **Edge case:** cây rỗng, một nút, hai nút bằng nhau/khác nhau, cây lệch trái/phải toàn một giá trị, cây hoàn hảo toàn một giá trị, giá trị âm, đường gấp khúc ở giữa, đường không qua gốc.
3. **Phản ví dụ** cho các cách làm sai (mục 13).
4. **Đối chiếu nhiều input** bằng `assert`: oracle mọi-cặp (Cách 1) với Cách 2, 3, 4 và `Solution` trên hàng nghìn cây ngẫu nhiên nhỏ có seed (giá trị trong bảng chữ nhỏ để nhiều nút trùng); thêm kiểm tra độc lập "toàn một giá trị ⇒ đáp án = đường kính cây" (bằng hai lần BFS).
5. **Cây lớn và sâu:** `n = 10^4` ngẫu nhiên, chuỗi sâu 1000.

In [18]:
IMPLS = [lup_bruteforce, lup_dfs, lup_parent_val, lup_iterative, lambda r: Solution().longestUnivaluePath(r)]
NAMES = ["bruteforce", "dfs", "parent_val", "iterative", "Solution"]

shapes = [
    [5, 4, 5, 1, 1, None, 5], [1, 4, 5, 4, 4, None, 5], [], [7], [1, 1], [1, 2], [1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1], [1, None, 1, None, 1, None, 1], [1, 1, None, 1, None, 1],
    [-3, -3, -3, -3], [2, 2, 3, 2, None, None, 3], [1, 2, 2, 3, 3, 3, 3], [1, 2, 3, 4, 5, 6, 7],
    [1, 9, 2, 3, 3, None, None, 3, 3],
]
for values in shapes:
    root = build_tree(values)
    base = lup_bruteforce(root)                  # oracle mọi-cặp làm chuẩn
    for f, name in zip(IMPLS, NAMES):
        assert f(root) == base, (name, values, f(root), base)
    print(f"{str(values):<40} -> {base}")
print(len(shapes), "hình dạng cây: cả 5 cách khớp oracle")

[5, 4, 5, 1, 1, None, 5]                 -> 2
[1, 4, 5, 4, 4, None, 5]                 -> 2
[]                                       -> 0
[7]                                      -> 0
[1, 1]                                   -> 1
[1, 2]                                   -> 0
[1, 1, 1]                                -> 2
[1, 1, 1, 1, 1, 1, 1]                    -> 4
[1, None, 1, None, 1, None, 1]           -> 3
[1, 1, None, 1, None, 1]                 -> 3
[-3, -3, -3, -3]                         -> 3
[2, 2, 3, 2, None, None, 3]              -> 2
[1, 2, 2, 3, 3, 3, 3]                    -> 0
[1, 2, 3, 4, 5, 6, 7]                    -> 0
[1, 9, 2, 3, 3, None, None, 3, 3]        -> 2
15 hình dạng cây: cả 5 cách khớp oracle


Với các ca ở trên ta lấy **oracle mọi-cặp** (định nghĩa thuần túy, không dùng DP) làm chuẩn rồi so các cách còn lại. Cell kế tiếp **ép cứng** những ca ta đã tự tính tay chắc chắn.

In [19]:
# Kiểm tra cứng các ca đã tính tay chắc chắn
sure = [
    ([5, 4, 5, 1, 1, None, 5], 2), ([1, 4, 5, 4, 4, None, 5], 2), ([], 0), ([7], 0), ([1, 1], 1), ([1, 2], 0),
    ([1, 1, 1], 2), ([1, 1, 1, 1, 1, 1, 1], 4), ([1, None, 1, None, 1, None, 1], 3), ([1, 2, 3, 4, 5, 6, 7], 0),
]
for values, expected in sure:
    for f, name in zip(IMPLS, NAMES):
        assert f(build_tree(values)) == expected, (name, values)
print(len(sure), "ca tính tay: cả 5 cách đều đúng")

10 ca tính tay: cả 5 cách đều đúng


In [20]:
# Đối chiếu ngẫu nhiên, cây NHỎ (có seed): oracle mọi-cặp vs các cách khác
rng = random.Random(687)
for trial in range(4000):
    n = rng.randint(1, 14)
    hi = rng.choice([1, 2, 3])                           # ít giá trị khác nhau -> nhiều chuỗi cùng giá trị
    values = random_tree_list(rng, n, 1, hi)
    root = build_tree(values)
    base = lup_bruteforce(root)
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        assert f(root) == base, (name, values, f(root), base)
print("4000 cây ngẫu nhiên nhỏ: mọi cách khớp oracle mọi-cặp")


# Kiểm tra độc lập: nếu MỌI nút cùng giá trị thì đáp án = đường kính cây (số cạnh dài nhất), tính bằng 2 lần BFS
def diameter_edges(root):
    adj, stack = {}, [root]
    adj[root] = []
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                adj[node].append(child)
                adj[child] = [node]
                stack.append(child)

    def farthest(src):
        dist, q, last = {src: 0}, deque([src]), src
        while q:
            u = q.popleft()
            last = u
            for v in adj[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return last, dist[last]

    far, _ = farthest(root)
    return farthest(far)[1]


for _ in range(1500):
    n = rng.randint(1, 40)
    root = build_tree(random_tree_list(rng, n, 5, 5))     # mọi giá trị đều 5
    d = diameter_edges(root)
    for f in IMPLS[1:]:
        assert f(root) == d
print("toàn một giá trị: đáp án = đường kính cây (kiểm tra bằng BFS độc lập): OK")

4000 cây ngẫu nhiên nhỏ: mọi cách khớp oracle mọi-cặp
toàn một giá trị: đáp án = đường kính cây (kiểm tra bằng BFS độc lập): OK


In [21]:
# Cây LỚN và SÂU đúng ràng buộc
def chain(n, val=1, side="left"):
    """Cây lệch n nút, mọi nút cùng giá trị."""
    root = TreeNode(val)
    cur = root
    for _ in range(n - 1):
        nxt = TreeNode(val)
        setattr(cur, side, nxt)
        cur = nxt
    return root


deep = chain(1000)                                       # độ sâu 1000 (đúng cận đề)
for f, name in zip(IMPLS[1:], NAMES[1:]):
    assert f(deep) == 999, name
print("chuỗi sâu 1000 toàn một giá trị -> 999: OK (cả 4 cách)")

zig = TreeNode(4)
cur = zig
for i in range(999):                                    # zig-zag trái/phải: vẫn là đường đi hợp lệ
    nxt = TreeNode(4)
    if i % 2 == 0:
        cur.left = nxt
    else:
        cur.right = nxt
    cur = nxt
assert all(f(zig) == 999 for f in IMPLS[1:])

rng = random.Random(1)
big = []
for hi in (1, 2, 5, 1000):
    root = build_tree(random_tree_list(rng, 10 ** 4, 1, hi))
    big.append((hi, root))
for hi, root in big:
    res = {}
    for f, name in zip(IMPLS[1:], NAMES[1:]):
        t0 = time.perf_counter()
        res[name] = f(root)
        res[name + "_t"] = time.perf_counter() - t0
    assert res["dfs"] == res["parent_val"] == res["iterative"] == res["Solution"]
    print(f"n=10^4, giá trị 1..{hi:<4} đáp án {res['dfs']:>4}  dfs {res['dfs_t']:.4f}s  iter {res['iterative_t']:.4f}s")

chuỗi sâu 1000 toàn một giá trị -> 999: OK (cả 4 cách)


n=10^4, giá trị 1..1    đáp án   62  dfs 0.0054s  iter 0.0053s
n=10^4, giá trị 1..2    đáp án   17  dfs 0.0045s  iter 0.0050s
n=10^4, giá trị 1..5    đáp án    9  dfs 0.0044s  iter 0.0050s
n=10^4, giá trị 1..1000 đáp án    1  dfs 0.0049s  iter 0.0054s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm **nút** thay vì **cạnh**

Đề: độ dài đường đi = **số cạnh**. Nếu tính `l + r + 1` (đếm nút) hoặc để lá trả `1`, kết quả luôn lớn hơn đáp án đúng `1`.

In [22]:
def wrong_nodes(root):
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        l = l + 1 if node.left and node.left.val == node.val else 0
        r = r + 1 if node.right and node.right.val == node.val else 0
        ans = max(ans, l + r + 1)                   # SAI: +1 đếm nút
        return max(l, r)

    dfs(root)
    return ans


print(wrong_nodes(ex1), "(đúng: 2)")
print(wrong_nodes(TreeNode(7)), "(đúng: 0 — cây một nút)")
print(wrong_nodes(build_tree([1, 2, 3])), "(đúng: 0)")

3

 (đúng: 2)
1 (đúng: 0 — cây một nút)
1 (đúng: 0)


**Sửa:** `ans = max(ans, l + r)` (số cạnh); cây một nút cho `0`.

### Lỗi 2: bỏ qua đường đi **gấp khúc** (chỉ trả `max(l, r)` và `ans = max(ans, max(l, r))`)

Chỉ xét chuỗi đi thẳng xuống (như đường từ gốc đến lá) bỏ lỡ đường `trái–đỉnh–phải`. Ví dụ 2 cho `1` thay vì `2`.

In [23]:
def wrong_no_bend(root):
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        l = l + 1 if node.left and node.left.val == node.val else 0
        r = r + 1 if node.right and node.right.val == node.val else 0
        ans = max(ans, max(l, r))                   # SAI: phải là l + r
        return max(l, r)

    dfs(root)
    return ans


print(wrong_no_bend(ex2), "(đúng: 2)")
print(wrong_no_bend(build_tree([1, 1, 1])), "(đúng: 2)")

1 (đúng: 2)
1 (đúng: 2)


**Sửa:** `ans = max(ans, l + r)`.

### Lỗi 3: quên kiểm tra **giá trị con bằng giá trị cha**

Nếu cộng `+1` cho mọi con có mặt (không kiểm `==`), ta đo **đường đi dài nhất bất kể giá trị** (chính là đường kính cây), không phải đường cùng giá trị.

In [24]:
def wrong_no_equal_check(root):
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        l = l + 1 if node.left else 0               # SAI: thiếu `and node.left.val == node.val`
        r = r + 1 if node.right else 0
        ans = max(ans, l + r)
        return max(l, r)

    dfs(root)
    return ans


print(wrong_no_equal_check(build_tree([1, 2, 3, 4, 5, 6, 7])), "(đúng: 0 — không có hai nút kề nhau cùng giá trị)")
print(wrong_no_equal_check(ex1), "(đúng: 2)")

4 (đúng: 0 — không có hai nút kề nhau cùng giá trị)
4 (đúng: 2)


**Sửa:** chỉ tăng khi con tồn tại **và** `node.left.val == node.val` (hoặc dùng `parent_val` như Cách 3).

### Lỗi 4: trả `l + r` lên cho cha (biến đường đi thành "cây hai nhánh")

Đường đi không được rẽ nhánh: cha chỉ nối được **một** trong hai nhánh. Trả `l + r` làm cha cộng thêm cả hai nhánh của con.

In [25]:
def wrong_return_sum(root):
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        l = l + 1 if node.left and node.left.val == node.val else 0
        r = r + 1 if node.right and node.right.val == node.val else 0
        ans = max(ans, l + r)
        return l + r                                # SAI: phải là max(l, r)

    dfs(root)
    return ans


perfect = build_tree([1] * 7)
print(wrong_return_sum(perfect), "(đúng: 4)")
print(lup_dfs(perfect))

6 (đúng: 4)
4


**Vì sao sai:** ở cây hoàn hảo 7 nút toàn `1`, các lá trả `0`; nút giữa trả `l + r = 2` (cả hai nhánh), gốc cộng `2+1` và `2+1` ⇒ `6`, thành "số cạnh của cả cây". **Sửa:** trả `max(l, r)`.

### Lỗi 5: chỉ tính đường đi **qua gốc**

Đề nói rõ đường đi **có thể không qua gốc**. Nếu chỉ tính `down(root.left) + down(root.right)` tại gốc, ta bỏ lỡ chuỗi nằm sâu trong một nhánh.

In [26]:
def wrong_root_only(root):
    def down(node, val):                            # số cạnh chuỗi cùng giá trị val đi xuống từ con của gốc
        if node is None or node.val != val:
            return 0
        return 1 + max(down(node.left, val), down(node.right, val))

    if root is None:
        return 0
    return down(root.left, root.val) + down(root.right, root.val)        # SAI: chỉ xét đỉnh = gốc


t = build_tree([1, 9, 2, 3, 3, None, None, 3, 3])          # chuỗi 3-3-3 nằm sâu bên trái, không qua gốc
print(wrong_root_only(t), "(đúng:", lup_dfs(t), ")")

0 (đúng: 2 )


**Sửa:** duyệt **mọi** nút làm đỉnh (DFS cập nhật `ans` ở mỗi nút).

**Checklist chống lỗi:**

- [ ] Đáp án đếm **cạnh**: `l + r` (không cộng thêm 1).
- [ ] Cập nhật `ans` bằng **`l + r`** ở **mọi** nút; trả **`max(l, r)`** lên cha.
- [ ] Chỉ nối con khi **giá trị con == giá trị nút**.
- [ ] Cây rỗng trả `0`.
- [ ] Nghĩ tới giới hạn đệ quy với cây sâu (hoặc dùng bản lặp).

## 14. Mẹo & nhận dạng pattern

- **"Đường đi dài nhất trong cây thoả điều kiện"** (đường kính, tổng lớn nhất, cùng giá trị...) ⇒ **Tree DP hậu thứ tự**: mỗi nút lấy thông tin hai con, **cập nhật đáp án bằng cách nối hai nhánh**, nhưng **chỉ trả lên một nhánh**.
- **Mẫu chung:** `dfs(node)` trả "tốt nhất khi đi **xuống** một nhánh từ node"; biến toàn cục `ans` lưu "tốt nhất khi **gấp khúc tại node**".
- **Phân biệt cạnh và nút** ngay từ đề (độ dài đường đi = số cạnh).
- **Kiểm tra điều kiện giữa cha và con** (bằng nhau, hơn kém 1...) ngay khi nối nhánh.
- **Cây sâu** ⇒ `sys.setrecursionlimit` hoặc bản lặp với thứ tự BFS đảo ngược/ngăn xếp.
- **Dựng/đọc cây từ list:** `build_tree` dùng hàng đợi; ghi nhớ `None` không có con trong danh sách.
- Bài họ hàng: 543 (Diameter of Binary Tree), 124 (Binary Tree Maximum Path Sum), 250 (Count Univalue Subtrees, Premium), 298/549 (Longest Consecutive Sequence), 1372 (Longest ZigZag Path).

## 15. Biến thể

1. **Bài 543 — Diameter of Binary Tree:** cùng khung, bỏ điều kiện "bằng giá trị" (mọi con đều nối): `ans = max(ans, l + r)` với `l, r` là chiều cao hai con + 1.
2. **Bài 124 — Maximum Path Sum:** thay "độ dài" bằng tổng giá trị; chỉ nối nhánh khi tổng nhánh dương.
3. **Bài 250 — Count Univalue Subtrees:** đếm số cây con mà mọi nút cùng giá trị (xem bài tập 5).
4. **Đường đi đơn điệu/liên tiếp** (298): điều kiện nối là `con = cha + 1`.
5. **Trả về chính đường đi** (danh sách nút), không chỉ độ dài: lưu thêm "nút cuối" của chuỗi dài nhất ở mỗi nhánh.

Minh hoạ nhanh biến thể 1 (đường kính) bằng đúng khung của bài này:

In [27]:
def diameter_dp(root):
    """Biến thể 1 (bài 543): giống lup_dfs nhưng con nào cũng nối (bỏ điều kiện bằng giá trị)."""
    ans = 0

    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        l = dfs(node.left)
        r = dfs(node.right)
        l = l + 1 if node.left else 0
        r = r + 1 if node.right else 0
        ans = max(ans, l + r)
        return max(l, r)

    dfs(root)
    return ans


print(diameter_dp(ex1), diameter_dp(build_tree([1, 2, 3, 4, 5])))       # 4 3
rng = random.Random(15)
for _ in range(500):
    root = build_tree(random_tree_list(rng, rng.randint(1, 30), 1, 3))
    assert diameter_dp(root) == diameter_edges(root)                     # khớp kiểm tra BFS độc lập
print("đường kính (DP) khớp BFS độc lập: OK")

4 3
đường kính (DP) khớp BFS độc lập: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp nút | tìm đường đi giữa mọi cặp, kiểm tra cùng giá trị | `O(n² · h)` | `O(n)` | oracle cho cây rất nhỏ |
| 2. DFS trả số cạnh | hậu thứ tự: `ans = max(l + r)`, trả `max(l, r)` | `O(n)` | `O(h)` | **nộp bài** |
| 3. DFS có `parent_val` | giống Cách 2 nhưng đếm nút và kiểm tra khớp cha ở đầu hàm | `O(n)` | `O(h)` | muốn mã gọn, kiểm tra điều kiện một chỗ |
| 4. Bản lặp (BFS đảo ngược) | xử lý con trước cha bằng `reversed(order)` | `O(n)` | `O(n)` | cây rất sâu, tránh giới hạn đệ quy |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được gọi, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Dự đoán: (1) `tree_to_list(build_tree([1, None, 2, 3]))`; (2) `Solution().longestUnivaluePath(build_tree([1, 1, 1, None, 1]))`; (3) `Solution().longestUnivaluePath(build_tree([1, 2, 3]))`. Ghi list `bai1_du_doan`.

In [28]:
# TODO Bài 1: bai1_du_doan = [..., ..., ...]
bai1_du_doan = None

### Bài 2 (tính tay)

Cây hoàn hảo 3 tầng `[1,1,1,1,1,1,1]` (7 nút, toàn `1`). Đường đi cùng giá trị dài nhất có bao nhiêu cạnh? Nếu đổi giá trị gốc thành `2` (`[2,1,1,1,1,1,1]`) thì sao? Ghi `(a, b)` vào `bai2`.

In [29]:
# TODO Bài 2: bai2 = (a, b)
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành hàm `dfs` bên trong lớp (đã có khung): trả số cạnh chuỗi cùng giá trị dài nhất đi xuống, đồng thời cập nhật `self.ans`.

In [30]:
class LUPTodo:
    def longestUnivaluePath(self, root):
        self.ans = 0

        def dfs(node):
            # TODO: nếu node rỗng trả 0; l, r = dfs(trái), dfs(phải);
            # tăng l/r thêm 1 nếu con tồn tại và bằng node.val, ngược lại 0;
            # cập nhật self.ans bằng l + r; trả max(l, r)
            pass

        dfs(root)
        return self.ans

### Bài 4 (tự viết lại)

Không nhìn mục 9, tự viết hàm `longest_univalue_path(root)` theo Cách 3 (có tham số `parent_val`, đếm nút).

In [31]:
# TODO Bài 4: tự viết lại
def longest_univalue_path(root):
    pass

### Bài 5 (biến thể / mini challenge)

Bài 250 (Count Univalue Subtrees): viết `count_unival_subtrees(root)` đếm số **cây con** (mỗi nút cùng toàn bộ hậu duệ của nó) mà **mọi nút có cùng giá trị**. Ví dụ `[5,1,5,5,5,None,5]` ⇒ `4`.

In [32]:
# TODO Bài 5
def count_unival_subtrees(root):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `build_tree` rồi `tree_to_list` trả lại đúng danh sách vào (cắt `None` thừa cuối).
- Mức 2: `[1,1,1,None,1]`: con trái của gốc có con phải `1`; đường `1–1–1–1`.
- Mức 3: `[[1, None, 2, 3], 3, 0]`.

**Bài 2**
- Mức 1: cây hoàn hảo 3 tầng: đường dài nhất là lá–giữa–gốc–giữa–lá.
- Mức 2: đổi gốc thành `2` thì gốc không nối được với con.
- Mức 3: `(4, 2)` — với gốc `2`: mỗi cây con 3 nút toàn `1` cho đường lá–giữa–lá = 2 cạnh.

**Bài 3**
- Mức 1: lá phải trả `0`.
- Mức 2: `l = l + 1 if node.left and node.left.val == node.val else 0`.
- Mức 3: `self.ans = max(self.ans, l + r)`; `return max(l, r)`.

**Bài 4**
- Mức 1: hàm `dfs(node, parent_val)` trả số **nút** của chuỗi khớp.
- Mức 2: `l = dfs(node.left, node.val)`, `r = dfs(node.right, node.val)`; `ans = max(ans, l + r)`.
- Mức 3: `return 1 + max(l, r) if node.val == parent_val else 0`; gọi `dfs(root, None)`.

**Bài 5**
- Mức 1: một cây con là "unival" nếu cả hai cây con của nó là unival và (nếu tồn tại) có giá trị bằng nút.
- Mức 2: hàm `dfs(node)` trả `True/False` (cây con gốc `node` có unival không) và cộng vào bộ đếm khi `True`; nhớ **luôn gọi cả hai con** trước khi kết luận (đừng dùng `and` làm bỏ qua việc duyệt).
- Mức 3: `ok = left_ok and right_ok and (node.left is None or node.left.val == node.val) and (node.right is None or node.right.val == node.val)`; lá luôn `True`.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [33]:
# Bài 1
dap_an_1 = [tree_to_list(build_tree([1, None, 2, 3])),
            Solution().longestUnivaluePath(build_tree([1, 1, 1, None, 1])),
            Solution().longestUnivaluePath(build_tree([1, 2, 3]))]
print(dap_an_1)
assert dap_an_1 == [[1, None, 2, 3], 3, 0]

# Bài 2
bai2_dap_an = (Solution().longestUnivaluePath(build_tree([1] * 7)),
               Solution().longestUnivaluePath(build_tree([2, 1, 1, 1, 1, 1, 1])))
print(bai2_dap_an)
assert bai2_dap_an == (4, 2)

[[1, None, 2, 3], 3, 0]
(4, 2)


In [34]:
# Bài 3
class LUPTodo:
    def longestUnivaluePath(self, root):
        self.ans = 0

        def dfs(node):
            if node is None:
                return 0
            l, r = dfs(node.left), dfs(node.right)
            l = l + 1 if node.left and node.left.val == node.val else 0
            r = r + 1 if node.right and node.right.val == node.val else 0
            self.ans = max(self.ans, l + r)
            return max(l, r)

        dfs(root)
        return self.ans


for values, expected in sure:
    assert LUPTodo().longestUnivaluePath(build_tree(values)) == expected
print("Bài 3: OK")

# Bài 4
def longest_univalue_path(root):
    ans = 0

    def dfs(node, parent_val):
        nonlocal ans
        if node is None:
            return 0
        l = dfs(node.left, node.val)
        r = dfs(node.right, node.val)
        ans = max(ans, l + r)
        return 1 + max(l, r) if node.val == parent_val else 0

    dfs(root, None)
    return ans


for values, expected in sure:
    assert longest_univalue_path(build_tree(values)) == expected
rng = random.Random(4)
for _ in range(1000):
    root = build_tree(random_tree_list(rng, rng.randint(1, 14), 1, rng.choice([1, 2, 3])))
    assert longest_univalue_path(root) == lup_bruteforce(root)
print("Bài 4: OK")

Bài 3: OK


Bài 4: OK


In [35]:
# Bài 5
def count_unival_subtrees(root):
    count = 0

    def dfs(node):
        nonlocal count
        if node is None:
            return True
        left_ok = dfs(node.left)                 # luôn duyệt cả hai con
        right_ok = dfs(node.right)
        ok = (left_ok and right_ok
              and (node.left is None or node.left.val == node.val)
              and (node.right is None or node.right.val == node.val))
        if ok:
            count += 1
        return ok

    dfs(root)
    return count


def unival_oracle(root):
    """Oracle: với mỗi nút, gom mọi giá trị của cây con rồi kiểm tra có đúng một giá trị."""
    def values(node):
        return [] if node is None else [node.val] + values(node.left) + values(node.right)

    def walk(node):
        if node is None:
            return 0
        return (1 if len(set(values(node))) == 1 else 0) + walk(node.left) + walk(node.right)

    return walk(root)


assert count_unival_subtrees(build_tree([5, 1, 5, 5, 5, None, 5])) == 4
assert count_unival_subtrees(build_tree([])) == 0 and count_unival_subtrees(build_tree([1, 1, 1])) == 3
rng = random.Random(5)
for _ in range(1500):
    root = build_tree(random_tree_list(rng, rng.randint(1, 15), 1, rng.choice([1, 2])))
    assert count_unival_subtrees(root) == unival_oracle(root)
print("Bài 5: OK")

Bài 5: OK

## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "Tree DP: nối hai nhánh để cập nhật đáp án, trả một nhánh lên cha":**

```python
class Solution:
    def longestUnivaluePath(self, root: Optional[TreeNode]) -> int:
        def dfs(node):
            if node is None:
                return 0
            l, r = dfs(node.left), dfs(node.right)
            l = l + 1 if node.left and node.left.val == node.val else 0   # chỉ nối khi cùng giá trị
            r = r + 1 if node.right and node.right.val == node.val else 0
            nonlocal ans
            ans = max(ans, l + r)                                         # đường gấp khúc tại node (số CẠNH)
            return max(l, r)                                              # chỉ trả MỘT nhánh lên cha

        ans = 0
        dfs(root)
        return ans
```

**Checklist trước khi submit:**

- [ ] `ans` đo bằng **cạnh**: `l + r` (không `+ 1`)?
- [ ] Cập nhật `ans` bằng `l + r` ở **mọi** nút (đường đi có thể không qua gốc, có thể gấp khúc)?
- [ ] Trả `max(l, r)` lên cha (không `l + r`)?
- [ ] Chỉ nối con khi **giá trị con == giá trị nút**?
- [ ] `nonlocal ans` (hoặc `self.ans` được reset mỗi lần gọi) để không giữ trạng thái giữa các test?
- [ ] Cây rỗng trả `0`; cây rất sâu: cân nhắc giới hạn đệ quy hoặc bản lặp?
- [ ] Đã test: rỗng, một nút, lệch, hoàn hảo toàn một giá trị, gấp khúc ở giữa, đường không qua gốc?
- [ ] Đã đối chiếu oracle mọi-cặp trên cây ngẫu nhiên nhỏ có seed?